In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class FeatureEngineeringConfig:
    root_dir: Path
    input_data_path: Path
    output_data_path: Path
    feature_report_path: Path
    original_feature_count: int

In [6]:
from src.ride_demand_forecasting_and_marketplace_optimization.constants import *
from src.ride_demand_forecasting_and_marketplace_optimization.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_feature_engineering_config(self) -> FeatureEngineeringConfig:

        config = self.config.feature_engineering

        create_directories(
            [config.root_dir]
        )

        feature_engineering_config = FeatureEngineeringConfig(

            root_dir=Path(config.root_dir),

            input_data_path=Path(config.input_data_path),

            output_data_path=Path(config.output_data_path),

            feature_report_path=Path(config.feature_report_path),
            original_feature_count=config.original_feature_count
        )    

        return feature_engineering_config

In [8]:
import json
import numpy as np
import pandas as pd
from src.ride_demand_forecasting_and_marketplace_optimization import logger
from src.ride_demand_forecasting_and_marketplace_optimization.utils.common import save_json

In [9]:
class FeatureEngineering:
    """
    Production Grade Feature Engineering
    Ride Demand Forecasting &
    Marketplace Optimization
    """

    def __init__(self,config: FeatureEngineeringConfig):

        self.config = config

        self.data = pd.read_parquet(
            self.config.input_data_path
        )

        self.feature_report = {}

        logger.info(
            f"Dataset Loaded "
            f"Shape={self.data.shape}"
        )

    # =====================================================
    # TIME FEATURES
    # =====================================================

    def create_time_features(self):

        self.data["timestamp"] = pd.to_datetime(
            self.data["timestamp"]
        )

        self.data["dayofyear"] = (
            self.data["timestamp"]
            .dt.dayofyear
        )

        self.data["is_month_start"] = (
            self.data["timestamp"]
            .dt.is_month_start
            .astype(int)
        )

        self.data["is_month_end"] = (
            self.data["timestamp"]
            .dt.is_month_end
            .astype(int)
        )

        self.data["is_quarter_end"] = (
            self.data["timestamp"]
            .dt.is_quarter_end
            .astype(int)
        )

        self.data["is_peak_hour"] = (
            self.data["hour"]
            .isin(
                [7, 8, 9, 17, 18, 19]
            )
            .astype(int)
        )

        self.data["is_business_hour"] = (
            self.data["hour"]
            .between(9, 18)
            .astype(int)
        )

        self.data["is_night"] = (
            (
                self.data["hour"] >= 22
            )
            |
            (
                self.data["hour"] <= 5
            )
        ).astype(int)

        logger.info(
            "Time features created"
        )

    # =====================================================
    # CYCLICAL FEATURES
    # =====================================================

    def create_cyclical_features(self):

        self.data["hour_sin"] = np.sin(
            2 *
            np.pi *
            self.data["hour"] /
            24
        )

        self.data["hour_cos"] = np.cos(
            2 *
            np.pi *
            self.data["hour"] /
            24
        )

        self.data["dayofweek_sin"] = np.sin(
            2 *
            np.pi *
            self.data["dayofweek"] /
            7
        )

        self.data["dayofweek_cos"] = np.cos(
            2 *
            np.pi *
            self.data["dayofweek"] /
            7
        )

        self.data["month_sin"] = np.sin(
            2 *
            np.pi *
            self.data["month"] /
            12
        )

        self.data["month_cos"] = np.cos(
            2 *
            np.pi *
            self.data["month"] /
            12
        )

        logger.info(
            "Cyclical features created"
        )

    # =====================================================
    # DEMAND LAGS
    # =====================================================

    def create_demand_lag_features(self):

        self.data.sort_values(
            [
                "zone_id",
                "timestamp"
            ],
            inplace=True
        )

        lag_hours = [
            1,
            2,
            3,
            6,
            12,
            24,
            168
        ]

        for lag in lag_hours:

            self.data[
                f"ride_requests_lag_{lag}"
            ] = (
                self.data
                .groupby("zone_id")
                ["ride_requests"]
                .shift(lag)
            )

        logger.info(
            "Demand lag features created"
        )

    # =====================================================
    # SUPPLY LAGS
    # =====================================================

    def create_supply_lag_features(self):

        for lag in [1, 24]:

            self.data[
                f"active_drivers_lag_{lag}"
            ] = (
                self.data
                .groupby("zone_id")
                ["active_drivers"]
                .shift(lag)
            )

            self.data[
                f"available_drivers_lag_{lag}"
            ] = (
                self.data
                .groupby("zone_id")
                ["available_drivers"]
                .shift(lag)
            )

        logger.info(
            "Supply lag features created"
        )

    # =====================================================
    # ROLLING FEATURES
    # =====================================================

    def create_rolling_features(self):

        grp = (
            self.data
            .groupby("zone_id")
            ["ride_requests"]
        )

        self.data[
            "ride_requests_roll_mean_3"
        ] = (
            grp.transform(
                lambda x:
                x.shift(1)
                .rolling(3)
                .mean()
            )
        )

        self.data[
            "ride_requests_roll_mean_6"
        ] = (
            grp.transform(
                lambda x:
                x.shift(1)
                .rolling(6)
                .mean()
            )
        )

        self.data[
            "ride_requests_roll_mean_24"
        ] = (
            grp.transform(
                lambda x:
                x.shift(1)
                .rolling(24)
                .mean()
            )
        )

        self.data[
            "ride_requests_roll_std_24"
        ] = (
            grp.transform(
                lambda x:
                x.shift(1)
                .rolling(24)
                .std()
            )
        )

        logger.info(
            "Rolling features created"
        )

    # =====================================================
    # DEMAND TREND FEATURES
    # =====================================================

    def create_demand_trend_features(self):

        self.data["demand_growth_24h"] = (
            (
                self.data[
                    "ride_requests_lag_1"
                ]
                -
                self.data[
                    "ride_requests_lag_24"
                ]
            )
            /
            (
                self.data[
                    "ride_requests_lag_24"
                ]
                + 1
            )
        )

        self.data["demand_growth_week"] = (
            (
                self.data[
                    "ride_requests_lag_1"
                ]
                -
                self.data[
                    "ride_requests_lag_168"
                ]
            )
            /
            (
                self.data[
                    "ride_requests_lag_168"
                ]
                + 1
            )
        )

        logger.info(
            "Demand trend features created"
        )

    # =====================================================
    # MARKETPLACE FEATURES
    # =====================================================

    def create_marketplace_features(self):

        self.data[
            "supply_demand_ratio"
        ] = (
            self.data["available_drivers"]
            /
            (
                self.data["ride_requests"]
                + 1
            )
        )

        self.data[
            "marketplace_gap"
        ] = (
            self.data["ride_requests"]
            -
            self.data["available_drivers"]
        )

        self.data[
            "driver_shortage"
        ] = (
            self.data[
                "marketplace_gap"
            ]
            .clip(lower=0)
        )

        self.data[
            "marketplace_pressure"
        ] = (
            self.data["ride_requests"]
            /
            (
                self.data[
                    "active_drivers"
                ]
                + 1
            )
        )

        logger.info(
            "Marketplace features created"
        )

    # =====================================================
    # DRIVER FEATURES
    # =====================================================

    def create_driver_efficiency_features(self):

        self.data[
            "rides_per_driver"
        ] = (
            self.data[
                "completed_rides"
            ]
            /
            (
                self.data[
                    "active_drivers"
                ]
                + 1
            )
        )

        self.data[
            "completion_rate"
        ] = (
            self.data[
                "completed_rides"
            ]
            /
            (
                self.data[
                    "ride_requests"
                ]
                + 1
            )
        )

        self.data[
            "cancellation_rate"
        ] = (
            self.data[
                "cancelled_rides"
            ]
            /
            (
                self.data[
                    "ride_requests"
                ]
                + 1
            )
        )

        logger.info(
            "Driver efficiency features created"
        )

    # =====================================================
    # SURGE FEATURES
    # =====================================================

    def create_surge_features(self):

        self.data[
            "surge_gap"
        ] = (
            self.data[
                "surge_multiplier"
            ]
            - 1
        )

        self.data[
            "surge_x_demand"
        ] = (
            self.data[
                "surge_multiplier"
            ]
            *
            self.data[
                "ride_requests"
            ]
        )

        self.data[
            "surge_x_traffic"
        ] = (
            self.data[
                "surge_multiplier"
            ]
            *
            self.data[
                "traffic_index"
            ]
        )

        logger.info(
            "Surge features created"
        )

    # =====================================================
    # WEATHER FEATURES
    # =====================================================

    def create_weather_features(self):

        self.data[
            "temperature_deviation"
        ] = (
            self.data[
                "temperature"
            ]
            -
            self.data[
                "temperature"
            ].mean()
        )

        self.data[
            "rain_intensity"
        ] = (
            self.data[
                "rainfall"
            ]
            *
            self.data[
                "humidity"
            ]
        )

        self.data[
            "weather_severity_score"
        ] = (
            self.data["rainfall"]
            +
            self.data["snowfall"] * 2
            +
            (
                self.data["wind_speed"]
                / 10
            )
        )

        logger.info(
            "Weather features created"
        )

    # =====================================================
    # TRAFFIC FEATURES
    # =====================================================

    def create_traffic_features(self):

        self.data[
            "traffic_x_demand"
        ] = (
            self.data[
                "traffic_index"
            ]
            *
            self.data[
                "ride_requests"
            ]
        )

        self.data[
            "traffic_pressure"
        ] = (
            self.data[
                "traffic_index"
            ]
            /
            (
                self.data[
                    "available_drivers"
                ]
                + 1
            )
        )

        logger.info(
            "Traffic features created"
        )

    # =====================================================
    # EVENT FEATURES
    # =====================================================

    def create_event_features(self):

        self.data["any_event"] = (

            self.data["holiday_flag"]

            +

            self.data["concert_flag"]

            +

            self.data["sports_event_flag"]

            +

            self.data["festival_flag"]

            > 0

        ).astype(int)

        self.data[
            "event_intensity"
        ] = (

            self.data["holiday_flag"]

            +

            self.data["concert_flag"]

            +

            self.data["sports_event_flag"]

            +

            self.data["festival_flag"]

        )

        logger.info(
            "Event features created"
        )

    # =====================================================
    # ECONOMIC FEATURES
    # =====================================================

    def create_economic_features(self):

        self.data[
            "fuel_cpi_ratio"
        ] = (
            self.data["fuel_price"]
            /
            (
                self.data["cpi"]
                + 1
            )
        )

        self.data[
            "economic_stress"
        ] = (
            self.data[
                "fuel_price"
            ]
            *
            self.data[
                "unemployment_rate"
            ]
        )

        logger.info(
            "Economic features created"
        )

    # =====================================================
    # GEOSPATIAL FEATURES
    # =====================================================

    def create_geospatial_features(self):

        self.data[
            "borough_avg_demand"
        ] = (
            self.data
            .groupby("borough")
            ["ride_requests"]
            .transform("mean")
        )

        self.data[
            "zone_avg_demand"
        ] = (
            self.data
            .groupby("zone_id")
            ["ride_requests"]
            .transform("mean")
        )

        logger.info(
            "Geospatial features created"
        )

    # =====================================================
    # REVENUE FEATURES
    # =====================================================

    def create_revenue_features(self):

        self.data[
            "revenue_per_request"
        ] = (
            self.data["revenue"]
            /
            (
                self.data[
                    "ride_requests"
                ]
                + 1
            )
        )

        self.data[
            "revenue_per_driver"
        ] = (
            self.data["revenue"]
            /
            (
                self.data[
                    "active_drivers"
                ]
                + 1
            )
        )

        self.data[
            "tip_percentage"
        ] = (
            self.data["avg_tip"]
            /
            (
                self.data["avg_fare"]
                + 1
            )
        )

        logger.info(
            "Revenue features created"
        )

    # =====================================================
    # FORECASTING FEATURES
    # =====================================================

    def create_forecasting_features(self):

        self.data[
            "same_hour_yesterday"
        ] = (
            self.data
            .groupby("zone_id")
            ["ride_requests"]
            .shift(24)
        )

        self.data[
            "same_hour_last_week"
        ] = (
            self.data
            .groupby("zone_id")
            ["ride_requests"]
            .shift(168)
        )

        logger.info(
            "Forecasting features created"
        )

    # =====================================================
    # ENTITY KEY
    # =====================================================

    def create_entity_key(self):

        self.data[
            "zone_timestamp_key"
        ] = (

            self.data["zone_id"]
            .astype(str)

            +

            "_"

            +

            self.data[
                "timestamp"
            ]
            .dt.strftime(
                "%Y%m%d%H"
            )
        )

        logger.info(
            "Entity key created"
        )

    # =====================================================
    # HANDLE GENERATED NANS
    # =====================================================

    def handle_generated_nans(self):

        numeric_cols = (
            self.data
            .select_dtypes(
                include=np.number
            )
            .columns
        )

        self.data[
            numeric_cols
        ] = (
            self.data[
                numeric_cols
            ]
            .fillna(0)
        )

        logger.info(
            "Generated NaNs handled"
        )

    # =====================================================
    # SAVE FEATURES
    # =====================================================

    def save_features(self):

        os.makedirs(
            os.path.dirname(
                self.config.output_data_path
            ),
            exist_ok=True
        )

        self.data.to_parquet(
            self.config.output_data_path,
            index=False
        )

        self.feature_report[
            "rows"
        ] = int(
            len(self.data)
        )

        self.feature_report[
            "columns"
        ] = int(
            len(self.data.columns)
        )

        self.feature_report[
            "generated_features"
        ] = int(
            len(self.data.columns)
            -
            self.config.original_feature_count
        )

        save_json(
            Path(
                self.config.feature_report_path
            ),
            self.feature_report
        )

        logger.info(
            "Feature engineering completed"
        )

    # =====================================================
    # MAIN PIPELINE
    # =====================================================

    def initiate_feature_engineering(self):

        logger.info(
            "Starting Feature Engineering"
        )

        self.create_time_features()

        self.create_cyclical_features()

        self.create_demand_lag_features()

        self.create_supply_lag_features()

        self.create_rolling_features()

        self.create_demand_trend_features()

        self.create_marketplace_features()

        self.create_driver_efficiency_features()

        self.create_surge_features()

        self.create_weather_features()

        self.create_traffic_features()

        self.create_event_features()

        self.create_economic_features()

        self.create_geospatial_features()

        self.create_revenue_features()

        self.create_forecasting_features()

        self.create_entity_key()

        self.handle_generated_nans()

        self.save_features()

        logger.info(
            "Feature Engineering Completed"
        )

        return (
            self.config.output_data_path
        )

In [10]:
try:
    config = ConfigurationManager()
    feature_engineering_config = config.get_feature_engineering_config()
    feature_engineering = FeatureEngineering(config = feature_engineering_config)
    output_data_pth = feature_engineering.initiate_feature_engineering()
except Exception as e:
    raise e

[2026-07-16 12:59:14,098: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-16 12:59:14,101: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-16 12:59:14,136: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-07-16 12:59:14,140: INFO: common: created directory at: artifacts]
[2026-07-16 12:59:14,142: INFO: common: created directory at: artifacts/feature_engineering]
[2026-07-16 12:59:14,481: INFO: 246742873: Dataset Loaded Shape=(174740, 46)]
[2026-07-16 12:59:14,485: INFO: 246742873: Starting Feature Engineering]
[2026-07-16 12:59:14,610: INFO: 246742873: Time features created]
[2026-07-16 12:59:14,670: INFO: 246742873: Cyclical features created]
[2026-07-16 12:59:14,786: INFO: 246742873: Demand lag features created]
[2026-07-16 12:59:14,832: INFO: 246742873: Supply lag features created]
[2026-07-16 12:59:15,000: INFO: 246742873: Rolling features created]
[2026-07-16 12:59:15,036: INFO: 246742873: Demand trend features cre

c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\numpy\core\_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
c:\Users\Hp\anaconda3\envs\demand\Lib\site-packages\pandas\core\nanops.py:731: RuntimeWarning: invalid value encountered in scalar divide
  the_mean = the_sum / count if count > 0 else np.nan
C:\Users\Hp\AppData\Local\Temp\ipykernel_2136\246742873.py:639: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("borough")


[2026-07-16 12:59:18,195: INFO: 246742873: Entity key created]
[2026-07-16 12:59:18,664: INFO: 246742873: Generated NaNs handled]
[2026-07-16 12:59:20,747: INFO: common: json file saved at: artifacts\feature_engineering\feature_report.json]
[2026-07-16 12:59:20,749: INFO: 246742873: Feature engineering completed]
[2026-07-16 12:59:20,749: INFO: 246742873: Feature Engineering Completed]


In [11]:
df = pd.read_parquet(output_data_pth)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174740 entries, 0 to 174739
Columns: 103 entries, timestamp to zone_timestamp_key
dtypes: category(2), datetime64[ns](1), float16(55), float32(5), float64(13), int16(4), int32(8), int8(14), object(1)
memory usage: 51.0+ MB


In [13]:
list(df.columns)

['timestamp',
 'zone_id',
 'zone_type',
 'latitude',
 'longitude',
 'borough',
 'ride_requests',
 'completed_rides',
 'cancelled_rides',
 'active_drivers',
 'available_drivers',
 'busy_drivers',
 'acceptance_rate',
 'utilization_rate',
 'rider_wait_time',
 'driver_wait_time',
 'surge_multiplier',
 'avg_fare',
 'avg_tip',
 'revenue',
 'avg_trip_distance',
 'avg_trip_duration',
 'traffic_index',
 'temperature',
 'feels_like',
 'humidity',
 'pressure',
 'visibility',
 'rainfall',
 'snowfall',
 'wind_speed',
 'holiday_flag',
 'concert_flag',
 'sports_event_flag',
 'festival_flag',
 'year',
 'quarter',
 'month',
 'week',
 'day',
 'hour',
 'dayofweek',
 'is_weekend',
 'fuel_price',
 'cpi',
 'unemployment_rate',
 'dayofyear',
 'is_month_start',
 'is_month_end',
 'is_quarter_end',
 'is_peak_hour',
 'is_business_hour',
 'is_night',
 'hour_sin',
 'hour_cos',
 'dayofweek_sin',
 'dayofweek_cos',
 'month_sin',
 'month_cos',
 'ride_requests_lag_1',
 'ride_requests_lag_2',
 'ride_requests_lag_3',
 'r